In [8]:
%load_ext autoreload
%autoreload 3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from __future__ import annotations

import numpy as np
from minitorch.tensor.tensor import Tensor

In [10]:
x_data = np.array([[1,2,3,4],[5,6,7,8]])
y_data = np.ones(shape=x_data.shape)
x = Tensor(x_data, requires_grad=True)
y = Tensor(y_data, requires_grad=True)
z = x @ y.transpose()

In [ ]:
z.backward()

In [ ]:
y.transpose()

Tensor(data=[[1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]],shape=(4, 2), grad_info= True)

In [ ]:
z.grad.shape, y.transpose().shape

((2, 2), (4, 2))

In [ ]:
y

Tensor(data=[[1. 1. 1. 1.]
 [1. 1. 1. 1.]],shape=(2, 4), grad_info= True)

In [ ]:
z.grad

array([[1., 1.],
       [1., 1.]], dtype=float32)

In [ ]:
x.grad.shape, y.grad.shape

((2, 4), (2, 4))

In [ ]:
x.mean(axis=0, keepdims=True)

Tensor(data=[[3. 4. 5. 6.]],shape=(1, 4), grad_info= True)

In [ ]:
x.mean(axis=1, keepdims=True)

Tensor(data=[[2.5]
 [6.5]],shape=(2, 1), grad_info= True)

In [11]:
from minitorch.nn.layers import Parameter

In [ ]:
class BatchNormalization:
    def __init__(self, dims: int, epsilon: float, momentum: float) -> None:
        self.dims = dims
        self.eps = epsilon
        self.momentum = momentum
        
        #* learneble parameters
        self.gamma = Parameter(np.ones(dims))
        self.beta = Parameter(np.zeros(dims))

In [16]:
#* hyperparameters
dimension = x.shape
epsilon = 1e-3
momentum = 0.1
training = True

#* learnable parameters
gamma = Tensor.ones(shape=dimension)
beta = Tensor.zeros(shape=dimension)

#* running mean and var (statistics not learnable)
running_mean = Tensor.zeros(shape=dimension)
running_var = Tensor.ones(shape=dimension)

if training:
    batch_mean = x.mean(axis=0, keepdims=True)
    batch_var = x.var(axis=0, keepdims=True)
    
    #* update the running statistics
    running_mean = Tensor(1 - momentum) * running_mean + Tensor(momentum) * batch_mean
    running_var = Tensor(1 - momentum) * running_mean + Tensor(momentum) * batch_var

    mean, var = batch_mean, batch_var
else:
    mean, var = running_mean, running_var
    
x_hat = (x - mean) / (var + epsilon) ** 0.5
out = gamma * x_hat + beta

out

Tensor(data=[[-0.70706259 -0.70706259 -0.70706259 -0.70706259]
 [ 0.70706259  0.70706259  0.70706259  0.70706259]],shape=(2, 4), grad_info= True)

In [ ]:
x.mean(axis=0, keepdims=True), x.var(axis=0, keepdims=True)

(Tensor(data=[[3. 4. 5. 6.]],shape=(1, 4), grad_info= True),
 Tensor(data=[[8. 8. 8. 8.]],shape=(1, 4), grad_info= True))

In [ ]:
out.mean(axis=0, keepdims=True), out.var(axis=0, keepdims=True)

(Tensor(data=[[0. 0. 0. 0.]],shape=(1, 4), grad_info= True),
 Tensor(data=[[0.99987502 0.99987502 0.99987502 0.99987502]],shape=(1, 4), grad_info= True))

In [43]:
(x - x.mean(axis=-1, keepdims=True)) / (x.var(axis=-1, keepdims=True) + epsilon) ** 0.5

Tensor(data=[[-1.16154659 -0.3871822   0.3871822   1.16154659]
 [-1.16154659 -0.3871822   0.3871822   1.16154659]],shape=(2, 4), grad_info= True)

In [ ]:
out.mean(axis=1, keepdims=True), out.var(axis=1, keepdims=True)

(Tensor(data=[[-0.70706259]
  [ 0.70706259]],shape=(2, 1), grad_info= True),
 Tensor(data=[[0.]
  [0.]],shape=(2, 1), grad_info= True))

In [34]:
from minitorch.nn.layers import LayerNormalization, BatchNormalization

In [35]:
ln = LayerNormalization(dim=4)
bn = BatchNormalization(dim=4)

norm1 = ln(x)
norm2 = bn(x)

In [45]:
norm1.mean(axis=1, keepdims=True), norm1.var(axis=1, keepdims=True)

(Tensor(data=[[-5.55111512e-17]
  [-5.55111512e-17]],shape=(2, 1), grad_info= True),
 Tensor(data=[[0.999994]
  [0.999994]],shape=(2, 1), grad_info= True))

In [46]:
norm1.mean(axis=0, keepdims=True), norm1.var(axis=0, keepdims=True)

(Tensor(data=[[-1.16189152 -0.38729717  0.38729717  1.16189152]],shape=(1, 4), grad_info= True),
 Tensor(data=[[0. 0. 0. 0.]],shape=(1, 4), grad_info= True))

In [47]:
norm2.mean(axis=0, keepdims=True), norm2.var(axis=0, keepdims=True)

(Tensor(data=[[0. 0. 0. 0.]],shape=(1, 4), grad_info= True),
 Tensor(data=[[0.99999875 0.99999875 0.99999875 0.99999875]],shape=(1, 4), grad_info= True))

In [48]:
norm2.mean(axis=1, keepdims=True), norm2.var(axis=1, keepdims=True)

(Tensor(data=[[-0.70710634]
  [ 0.70710634]],shape=(2, 1), grad_info= True),
 Tensor(data=[[0.]
  [0.]],shape=(2, 1), grad_info= True))